## 9. 权限系统与 Human-in-the-Loop

> 来源：[Configure permissions](https://code.claude.com/docs/en/agent-sdk/permissions)、[Handle approvals and user input](https://code.claude.com/docs/en/agent-sdk/user-input)


### 9.1 权限评估顺序（先记这条流水线）

Claude 每请求一次 tool，SDK 按固定顺序评估，**前面的步骤解决了，后面的就不再执行**：

```text
Hooks → Deny 规则 → Ask 规则 → Permission mode → Allow 规则 → can_use_tool 回调
```

1. **Hooks** 最先跑。hook 可以直接 deny；但 hook 返回 allow **不会跳过**后面的 deny/ask 规则。
2. **Deny 规则**（`disallowed_tools` + settings.json）匹配即阻断，**连 `bypassPermissions` 都拦得住**。
3. **Ask 规则** 匹配则强制落入 `can_use_tool` 询问；`AskUserQuestion` 等"必须与人交互"的 tool 也总是落到回调——但 `dontAsk` 模式下这类交互工具不落回调、**直接被 deny**（该模式从不询问），headless agent 里用不了 `AskUserQuestion`。
4. **Permission mode**：`bypassPermissions` 批准到这一步的一切；`acceptEdits` 批准文件操作；`plan` 把写操作强制引回回调（无视 allow 规则）。
5. **Allow 规则**（`allowed_tools` + settings.json）匹配即批准。
6. **`can_use_tool` 回调** 兜底；`dontAsk` 模式下跳过此步直接 deny。

拿一份具体配置走一遍。同一份 options 下，三次 tool 请求各停在流水线的哪一步：

```python
options = ClaudeAgentOptions(
    allowed_tools=["Read"],            # 第 5 步的 allow 规则
    disallowed_tools=["Bash(rm *)"],   # 第 2 步的 deny 规则（带范围）
    permission_mode="default",
    can_use_tool=gate,                 # 第 6 步的兜底回调
)

# Read("notes.md")      → 第 5 步 allow 规则命中，自动批准；gate 不会被调用
# Bash("rm -rf build")  → 第 2 步 deny 规则命中，直接阻断；gate 同样不会被调用
# Bash("ls")            → 前五步都没接住，落到第 6 步，gate 说了算
```

由这条流水线推出四个关键结论：

- **被前面任何一步批准的调用，永远到不了 `can_use_tool`**——上例里 `Read` 根本不经过 gate，写在回调里的检查对 `allowed_tools` 里的 tool 静默失效。必须每次调用都生效的逻辑，用 `PreToolUse` hook（它在整条流水线之前，连 `bypassPermissions` 都绕不过它的 deny）。
- **`allowed_tools` 不约束 `bypassPermissions`**：`allowed_tools=["Read"]` + `bypassPermissions` 仍然批准所有工具——所有调用在第 4 步就被放行，第 5 步的名单形同虚设。要在该模式下禁工具只能用 `disallowed_tools`（第 2 步，在模式之前）。
- **deny 规则两种写法语义不同**：裸名 `"Bash"` 把 tool 定义整个从 context 移除（Claude 根本看不见）；带范围 `"Bash(rm *)"` 保留 tool、只 deny 匹配的调用（上例里 `Bash("ls")` 照常往下走）。
- **通配符两侧不对称**：allow 侧 `"*"` 和 `"mcp__*"` **无效**——被忽略并产生启动警告、不批准任何东西，通配只能出现在字面 `mcp__<server>__` 前缀之后（如 `mcp__github__*`）；deny 侧支持全局通配——`disallowed_tools=["*"]` 移除全部工具定义，`"mcp__*"` 匹配全部 MCP 工具。

一个生效前提：settings.json 里的 allow/deny/ask 规则要参与评估，`setting_sources` 必须含 `"project"`（省略时默认含；显式传了列表就得自己带上，见 §20.1「三个 source 的语义」）——否则这些规则**静默失效**。

### 9.2 permission_mode 全表

| 模式 | 行为 | 适用 |
|---|---|---|
| `default` | 未被规则覆盖的 tool 触发 `can_use_tool`；没配回调则 deny | 交互应用 + 审批回调 |
| `acceptEdits` | 自动批准文件编辑与文件系统命令（`mkdir`/`touch`/`rm`/`rmdir`/`mv`/`cp`/`sed`），仅限 `cwd` 和 `add_dirs` 范围内；写受保护路径仍会提示 | 受信任的开发工作流 |
| `plan` | 只探索不改动；写操作永不自动批准，经回调询问 | 先规划再动手 |
| `dontAsk` | 从不询问：预批准的运行，其余直接 deny，回调不会被调用；交互类工具（`AskUserQuestion` 等）直接被 deny | 锁死的 headless agent |
| `bypassPermissions` | 一律放行（除 deny/ask 规则和 hooks）；Unix root 下不可用 | 沙箱 CI、隔离环境 |

（TypeScript 另有模型分类器审批的 `auto` 模式，Python 无。）

推荐配对：交互应用 `default` + 回调；开发机自主 agent `acceptEdits`；锁死型 agent `allowed_tools` + `dontAsk`；`bypassPermissions` 只留给容器/CI。**MCP tool 的授权优先用 `allowed_tools` 通配符（如 `mcp__github__*`），不要靠放宽 permission mode**——`acceptEdits` 不批 MCP tool，`bypassPermissions` 又批得太宽。

> [!warning] 子 agent 继承父级模式且不可覆盖
> 父级用 `bypassPermissions` / `acceptEdits` 时，全部子 agent 继承该模式，`AgentDefinition.permissionMode` 覆盖不了（§14.1「编程式定义」）——继承 `bypassPermissions` 等于放给子 agent 完全自主的系统访问，而子 agent 的 system prompt 可能比主 agent 更不受约束。显式 ask 规则仍可强制提示。

运行期可切模式：`await client.set_permission_mode("acceptEdits")`（`ClaudeSDKClient`），典型用法是先严后松。

### 9.3 `can_use_tool` 回调（HITL 的落点）

回调签名与返回值：

```python
async def can_use_tool(
    tool_name: str,                    # "Bash" / "Write" / "AskUserQuestion"...
    input_data: dict,                  # tool 入参，内容随 tool 而异
    context: ToolPermissionContext,    # 携带 suggestions（现成的权限更新建议）
) -> PermissionResultAllow | PermissionResultDeny: ...
```

- `PermissionResultAllow(updated_input=...)`：放行，还能**改写工具入参**（比如把写路径重定向到沙箱；Claude 不知道被改过）。
- `PermissionResultAllow(updated_input=..., updated_permissions=...)`：批准并持久化规则（从 `context.suggestions` 里挑 `destination == "localSettings"` 的条目回传，即"总是允许"；需 SDK ≥ 0.1.80）。
- `PermissionResultDeny(message=..., interrupt=...)`：拒绝并给模型一个理由——理由写得好，Claude 会换方案（如"用户不想删文件，问能否改成压缩归档"）。
- 还有一种**整体改道**：不批也不驳这次调用，直接用流式输入给 Claude 发一条全新指令接管方向（§16「流式输入」）。

因为回调是 `async`，可以在里面 `await` 一个 Future 挂起，等真人点"批准/拒绝"再返回——把"异步等人"伪装成一次同步权限判断，Human-in-the-Loop 的本质就是这个。回调可以无限期挂起等待；但要等的时间一旦超过进程本身的存活时间，就不能硬等——改用 hook 的 `defer` 决策先让进程退出，之后再 resume 接着处理（§11.3「回调签名与返回值」）。

> [!warning] Python 专属坑：回调需要流式输入 + dummy hook
> Python 里 `can_use_tool` 要求 **streaming mode**（`prompt` 传 async generator，或直接用 `ClaudeSDKClient`）。用 `query()` 时还需注册一个返回 `{"continue_": True}` 的 `PreToolUse` hook 保持流打开（注意键名带下划线），否则流会在回调触发前关闭。下面的示例即官方推荐写法。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
from claude_agent_sdk.types import (
    HookMatcher,
    PermissionResultAllow,
    PermissionResultDeny,
    ToolPermissionContext,
)


async def gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    # 例1：禁止危险删除，并给模型一个可行的替代方向
    if tool_name == "Bash" and "rm " in input_data.get("command", ""):
        return PermissionResultDeny(
            message="User doesn't want to delete files; compress them into an archive instead."
        )
    # 例2：把碰 config 的写操作重定向到沙箱（改写入参后放行，Claude 无感知）
    if tool_name in ("Write", "Edit") and "config" in input_data.get("file_path", ""):
        safe = f"./sandbox/{input_data['file_path']}"
        return PermissionResultAllow(updated_input={**input_data, "file_path": safe})
    # 例3（HITL 骨架）：真人审批
    #   fut = asyncio.get_running_loop().create_future()
    #   push_to_ui(tool_name, input_data, fut)   # 前端弹审批框，点击后 fut.set_result(bool)
    #   return PermissionResultAllow(updated_input=input_data) if await fut \
    #       else PermissionResultDeny(message="user denied")
    return PermissionResultAllow(updated_input=input_data)


# Python 必需的 workaround：dummy PreToolUse hook 保持流打开，否则回调不会被触发
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


# can_use_tool 要求流式输入：prompt 传 async generator 而非字符串
async def prompt_stream():
    yield {
        "type": "user",
        "message": {"role": "user", "content": "Update the app config file"},
    }


async def demo_gate():
    async for m in query(
        prompt=prompt_stream(),
        options=ClaudeAgentOptions(
            can_use_tool=gate,
            hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[dummy_hook])]},
        ),
    ):
        if isinstance(m, ResultMessage) and m.subtype == "success":
            print(m.result)


await demo_gate()

### 9.4 另一条 HITL 通道：`AskUserQuestion`

上面的 `can_use_tool` 是**你拦 Claude**；`AskUserQuestion` 是**Claude 主动问你**。它是个内建工具，Claude 觉得多个方案都合理、需要方向时调用（`plan` 模式下尤其常见）。它同样触发 `can_use_tool` 回调（`tool_name == "AskUserQuestion"`），你负责把问题呈现给真人、把选择回填——**问题和选项由 Claude 生成，你不能往这个流程里塞自己的问题**。

**输入格式**（`input_data["questions"]`，每次 1–4 个问题）：

| 字段 | 含义 |
|---|---|
| `question` | 完整问题文本 |
| `header` | 短标签（≤12 字符） |
| `options` | 2–4 个选项，各有 `label` 和 `description` |
| `multiSelect` | `true` 则可多选 |

**回填格式**：返回 `PermissionResultAllow(updated_input=...)`，其中 `updated_input` 必须包含**原样传回的 `questions`** 加上 `answers`——key 是问题文本，value 是所选 `label`（多选传 label 列表）。用户打了自由文本就直接放原文，不要放 "Other"：

```python
return PermissionResultAllow(
    updated_input={
        "questions": input_data.get("questions", []),
        "answers": {
            "How should I format the output?": "Summary",
            "Which sections should I include?": ["Introduction", "Conclusion"],
        },
    }
)
```

用户不按题回答、整体打了一段话时，放进与 `questions` 并列的顶层 `response` 字段——Claude 收到的是 "The user responded: …" 而非逐题答案。

三个限制：子 agent 内当前不可用 `AskUserQuestion`；`dontAsk` 模式下它直接被 deny（§9.1「权限评估顺序」 第 3 步）；如果用 `tools` 字段收窄了工具集，必须把 `"AskUserQuestion"` 加回名单，否则 Claude 无法提问。

两条通道最终都汇成"agent 停下等人输入"的状态。更复杂的交互（表单、多步向导、对接外部审批系统）用自定义工具实现（§10），那是控制力最强、实现成本也最高的一档。完整的终端问答实现（展示问题 → 收输入 → 数字选项或自由文本解析 → 回填）见下方 cell。

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions
from claude_agent_sdk.types import HookMatcher, PermissionResultAllow


def ask_user(questions: list) -> dict:
    """把 Claude 的问题呈现给真人：输入数字选选项，直接打字算自由回答。"""
    answers = {}
    for q in questions:
        print(f"\n[{q.get('header', '')}] {q['question']}")
        options = q.get("options", [])
        for i, opt in enumerate(options, 1):
            print(f"  {i}. {opt['label']} —— {opt.get('description', '')}")
        raw = input("> ").strip()
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            answers[q["question"]] = options[int(raw) - 1]["label"]
        else:
            answers[q["question"]] = raw  # 自由文本原样放入，不要写 "Other"
    return answers


async def gate(tool_name, input_data, context):
    if tool_name == "AskUserQuestion":
        return PermissionResultAllow(
            updated_input={
                "questions": input_data.get("questions", []),  # 原样传回
                "answers": ask_user(input_data.get("questions", [])),
            }
        )
    return PermissionResultAllow(updated_input=input_data)


async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}  # §9.3「can_use_tool 回调」 的 Python workaround：保持流打开


async def prompt_stream():
    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": "Plan a refactor of the auth module; ask me before choosing a direction.",
        },
    }


async def demo_ask_user_question():
    async for m in query(
        prompt=prompt_stream(),
        options=ClaudeAgentOptions(
            permission_mode="plan",  # plan 模式下 Claude 更常主动提问
            can_use_tool=gate,
            hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[dummy_hook])]},
        ),
    ):
        if hasattr(m, "result"):
            print(m.result)


await demo_ask_user_question()